# PHM08 (PHM 2008 Challenge) — Early Fault + RUL Quantiles (DS-TCN)

This notebook adapts the **same CMAPSS pipeline** to the **PHM08 Challenge** dataset:
- Sliding windows from multivariate time series
- Dual head: **Early-fault probability** + **RUL quantiles**
- Physics-guided monotonic constraint on the health-index proxy (q50)
- Full evaluation on a **held-out validation split from train.txt** (since PHM08 test labels are hidden)
- Saves plots (**PDF**) and metrics (**JSON/CSV**) into `results/phm08/`

**Files expected** (relative to project root):
- `dataset/train.txt`
- `dataset/test.txt`
- `dataset/final_test.txt`


In [1]:

# ============ Imports ============
import os, json, math, random, time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score, roc_curve, precision_recall_curve
)

import matplotlib.pyplot as plt

# ============ Reproducibility ============
def seed_all(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_all(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

# ============ Paths ============
ROOT = Path.cwd()
DATA_DIR = ROOT / "dataset"          # user: train.txt/test.txt/final_test.txt are here
RES_DIR  = ROOT / "results" / "phm08"
(RES_DIR / "plots").mkdir(parents=True, exist_ok=True)
(RES_DIR / "metrics").mkdir(parents=True, exist_ok=True)
(RES_DIR / "preds").mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR.resolve())
print("RES_DIR :", RES_DIR.resolve())


DEVICE: cuda
DATA_DIR: E:\Research\Aircraft_Maintenance\code_PHM2008\dataset
RES_DIR : E:\Research\Aircraft_Maintenance\code_PHM2008\results\phm08


## 0) IEEE-style plot settings (white background, clean for papers)


In [2]:

# ============ IEEE rcParams ============
plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 300,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "lines.linewidth": 2.0,
})


## 1) PHM08 parsing + correct RUL construction (train only)
PHM08 rows have **26 columns**: `unit, cycle, op1, op2, op3, s1..s21`.
Only `train.txt` is run-to-failure, so only train has true RUL.


In [3]:

PHM08_COLS = ["unit","cycle","op1","op2","op3"] + [f"s{i}" for i in range(1,22)]  # 5 + 21 = 26
FEAT_COLS  = ["op1","op2","op3"] + [f"s{i}" for i in range(1,22)]

def read_phm08_txt(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=r"\s+", header=None)
    df = df.iloc[:, :26]
    df.columns = PHM08_COLS
    df["unit"] = df["unit"].astype(int)
    df["cycle"] = df["cycle"].astype(int)
    return df

def compute_rul_train(df_train: pd.DataFrame, rul_cap: Optional[int] = None) -> pd.DataFrame:
    df = df_train.copy()
    max_cycle = df.groupby("unit")["cycle"].max().to_dict()
    df["RUL"] = df.apply(lambda r: float(max_cycle[int(r["unit"])] - int(r["cycle"])), axis=1)
    if rul_cap is not None:
        df["RUL"] = np.minimum(df["RUL"].values, float(rul_cap))
    return df

train_path = DATA_DIR / "train.txt"
test_path  = DATA_DIR / "test.txt"
final_path = DATA_DIR / "final_test.txt"

df_train_raw = read_phm08_txt(train_path)
df_test_raw  = read_phm08_txt(test_path)
df_final_raw = read_phm08_txt(final_path)

df_train = compute_rul_train(df_train_raw, rul_cap=None)

print("Train shape:", df_train.shape, "units:", df_train["unit"].nunique())
print("Test shape :", df_test_raw.shape, "units:", df_test_raw["unit"].nunique())
print("Final shape:", df_final_raw.shape, "units:", df_final_raw["unit"].nunique())

df_train.head()


Train shape: (45918, 27) units: 218
Test shape : (29820, 26) units: 218
Final shape: (55156, 26) units: 435


,unit,cycle,op1,op2,op3,s1,s2,s3,s4,s5,...,s13,s14,s15,s16,s17,s18,s19,s20,s21,RUL
0,1,1,10.0047,0.2501,20.0,489.05,604.13,1499.45,1309.95,10.52,...,2388.13,8120.83,8.6216,0.03,368,2319,100.0,28.58,17.1735,222.0
1,1,2,0.0015,0.0003,100.0,518.67,642.13,1584.55,1403.96,14.62,...,2388.15,8132.87,8.3907,0.03,391,2388,100.0,38.99,23.3619,221.0
2,1,3,34.9986,0.8401,60.0,449.44,555.42,1368.17,1122.49,5.48,...,2387.95,8063.84,9.3557,0.02,334,2223,100.0,14.83,8.8555,220.0
3,1,4,20.0031,0.7005,0.0,491.19,607.03,1488.44,1249.18,9.35,...,2388.07,8052.30,9.2231,0.02,364,2324,100.0,24.42,14.7832,219.0
4,1,5,42.0041,0.8405,40.0,445.00,549.52,1354.48,1124.32,3.91,...,2387.89,8083.67,9.2986,0.02,330,2212,100.0,10.99,6.4025,218.0


## 2) Sliding-window dataset construction + fault label
- Target RUL = RUL at window end
- Fault label = 1 if `RUL_end ≤ tau`


In [4]:

@dataclass
class WindowConfig:
    W: int = 30
    stride: int = 1
    tau: int = 30
    rul_cap: Optional[int] = None

def make_windows_for_units(df: pd.DataFrame, unit_ids: List[int], cfg: WindowConfig) -> Dict[str, np.ndarray]:
    W, stride = cfg.W, cfg.stride
    X_list, y_rul_list, y_fault_list, unit_list, endc_list = [], [], [], [], []
    for u in unit_ids:
        d = df[df["unit"] == u].sort_values("cycle").reset_index(drop=True)
        feats = d[FEAT_COLS].values.astype(np.float32)
        rul = d["RUL"].values.astype(np.float32)
        cycles = d["cycle"].values.astype(np.int32)

        n = len(d)
        if n < W:
            continue

        for i in range(0, n - W + 1, stride):
            x = feats[i:i+W]
            rul_end = float(rul[i+W-1])
            fault = 1.0 if rul_end <= cfg.tau else 0.0
            X_list.append(x)
            y_rul_list.append(rul_end)
            y_fault_list.append(fault)
            unit_list.append(int(u))
            endc_list.append(int(cycles[i+W-1]))

    return {
        "X": np.array(X_list, dtype=np.float32),
        "y_rul": np.array(y_rul_list, dtype=np.float32),
        "y_fault": np.array(y_fault_list, dtype=np.float32),
        "unit": np.array(unit_list, dtype=np.int32),
        "end_cycle": np.array(endc_list, dtype=np.int32),
    }

class WindowDataset(Dataset):
    def __init__(self, arrays: Dict[str, np.ndarray]):
        self.X = arrays["X"]
        self.y_rul = arrays["y_rul"]
        self.y_fault = arrays["y_fault"]
        self.unit = arrays["unit"]
        self.end_cycle = arrays["end_cycle"]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx])          # [W, F]
        y_rul = torch.tensor(self.y_rul[idx])
        y_fault = torch.tensor(self.y_fault[idx])
        unit = torch.tensor(self.unit[idx])
        endc = torch.tensor(self.end_cycle[idx])
        return x, y_rul, y_fault, unit, endc


## 3) Engine-wise train/val split + normalization (fit on train only)


In [5]:

@dataclass
class SplitConfig:
    val_ratio: float = 0.2
    seed: int = 42

def split_units_enginewise(units: List[int], cfg: SplitConfig) -> Tuple[List[int], List[int]]:
    rng = np.random.default_rng(cfg.seed)
    units = np.array(sorted(units), dtype=int)
    rng.shuffle(units)
    n_val = max(1, int(len(units) * cfg.val_ratio))
    val_units = sorted(units[:n_val].tolist())
    tr_units = sorted(units[n_val:].tolist())
    return tr_units, val_units

def fit_standardizer(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    mu = X.reshape(-1, X.shape[-1]).mean(axis=0)
    sd = X.reshape(-1, X.shape[-1]).std(axis=0) + 1e-6
    return mu.astype(np.float32), sd.astype(np.float32)

def apply_standardizer(X: np.ndarray, mu: np.ndarray, sd: np.ndarray) -> np.ndarray:
    return ((X - mu[None,None,:]) / sd[None,None,:]).astype(np.float32)

wcfg = WindowConfig(W=30, stride=1, tau=30, rul_cap=None)
scfg = SplitConfig(val_ratio=0.2, seed=42)

all_units = sorted(df_train["unit"].unique().tolist())
tr_units, va_units = split_units_enginewise(all_units, scfg)
print("Units:", len(all_units), "Train units:", len(tr_units), "Val units:", len(va_units))

train_arrays = make_windows_for_units(df_train, tr_units, wcfg)
val_arrays   = make_windows_for_units(df_train, va_units, wcfg)

mu, sd = fit_standardizer(train_arrays["X"])
train_arrays["X"] = apply_standardizer(train_arrays["X"], mu, sd)
val_arrays["X"]   = apply_standardizer(val_arrays["X"], mu, sd)

np.save(RES_DIR / "metrics" / "standardizer_mu.npy", mu)
np.save(RES_DIR / "metrics" / "standardizer_sd.npy", sd)

print("Train windows:", train_arrays["X"].shape, "Val windows:", val_arrays["X"].shape)
print("Pos rate (train):", float(train_arrays["y_fault"].mean()), "Pos rate (val):", float(val_arrays["y_fault"].mean()))


Units: 218 Train units: 175 Val units: 43
Train windows: (31477, 30, 24) Val windows: (8119, 30, 24)
Pos rate (train): 0.17234806716442108 Pos rate (val): 0.16418278217315674


## 4) Model: DS-TCN encoder + dual head


In [6]:

class DepthwiseSeparableTCNBlock(nn.Module):
    def __init__(self, channels: int, out_channels: int, kernel_size: int, dilation: int, dropout: float):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.dw = nn.Conv1d(channels, channels, kernel_size=kernel_size, dilation=dilation,
                            padding=padding, groups=channels, bias=False)
        self.pw = nn.Conv1d(channels, out_channels, kernel_size=1, bias=True)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        self.use_res = (channels == out_channels)

    def forward(self, x):
        y = self.dw(x)
        y = self.pw(y)
        y = self.act(y)
        y = self.drop(y)
        if self.use_res:
            y = y + x
        return y

class DS_TCN_Encoder(nn.Module):
    def __init__(self, in_channels: int, hidden: int = 64, kernel_size: int = 3, dilations=(1,2,4,8), dropout: float = 0.1):
        super().__init__()
        layers = []
        ch = in_channels
        for d in dilations:
            layers.append(DepthwiseSeparableTCNBlock(ch, hidden, kernel_size=kernel_size, dilation=d, dropout=dropout))
            ch = hidden
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)  # [B, hidden, T]

class DualHeadModel(nn.Module):
    def __init__(self, n_features: int, hidden: int = 64, dropout: float = 0.1):
        super().__init__()
        self.enc = DS_TCN_Encoder(in_channels=n_features, hidden=hidden, dropout=dropout)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fault_head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1)
        )
        self.rul_head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 3)  # q10,q50,q90
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)          # [B,F,W]
        z = self.enc(x)                 # [B,hidden,W]
        h = self.pool(z).squeeze(-1)    # [B,hidden]
        p_fault = torch.sigmoid(self.fault_head(h)).squeeze(-1)
        q = self.rul_head(h)
        # enforce ordering
        q10, q50, q90 = q[:,0], q[:,1], q[:,2]
        q50 = torch.maximum(q50, q10)
        q90 = torch.maximum(q90, q50)
        q = torch.stack([q10, q50, q90], dim=1)
        return {"p_fault": p_fault, "q": q, "z": z, "h": h}


## 5) Losses + monotonic constraint


In [7]:

@dataclass
class LossConfig:
    lambda_rul: float = 1.0
    lambda_fault: float = 1.0
    lambda_mono: float = 0.1
    focal_gamma: float = 2.0

ALPHAS = (0.1, 0.5, 0.9)

def pinball_loss(q: torch.Tensor, y: torch.Tensor, alphas=ALPHAS) -> torch.Tensor:
    losses = []
    for i, a in enumerate(alphas):
        e = y - q[:, i]
        losses.append(torch.maximum(a*e, (a-1)*e).mean())
    return sum(losses)

def focal_loss(p: torch.Tensor, y: torch.Tensor, alpha: float, gamma: float = 2.0, eps: float = 1e-6) -> torch.Tensor:
    p = torch.clamp(p, eps, 1-eps)
    pt = torch.where(y == 1, p, 1-p)
    w = torch.where(y == 1, torch.tensor(alpha, device=p.device), torch.tensor(1-alpha, device=p.device))
    return (-w * (1-pt)**gamma * torch.log(pt)).mean()

def monotonic_loss(q50_t: torch.Tensor, q50_t1: torch.Tensor) -> torch.Tensor:
    return F.relu(q50_t1 - q50_t).mean()

def batch_alpha_from_imbalance(y_fault: torch.Tensor) -> float:
    pos = float((y_fault > 0.5).float().mean().item())
    return float(np.clip(0.5 / max(pos, 1e-4), 0.5, 0.99))


## 6) Metrics + plotting helpers (PDF) and JSON saving


In [8]:

def compute_fault_metrics(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return {"auprc": float("nan"), "auroc": float("nan"), "f1@0.5": float("nan"), "recall@fpr1%": float("nan")}
    out = {}
    out["auprc"] = float(average_precision_score(y_true, y_prob))
    out["auroc"] = float(roc_auc_score(y_true, y_prob))
    out["f1@0.5"] = float(f1_score(y_true, (y_prob >= 0.5).astype(int)))
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    mask = fpr <= 0.01
    out["recall@fpr1%"] = float(np.max(tpr[mask])) if np.any(mask) else 0.0
    return out

def compute_rul_metrics(y_true: np.ndarray, q: np.ndarray) -> Dict[str, float]:
    q50 = q[:, 1]
    mae = float(np.mean(np.abs(q50 - y_true)))
    rmse = float(np.sqrt(np.mean((q50 - y_true)**2)))
    cov = float(np.mean((y_true >= q[:,0]) & (y_true <= q[:,2])))
    iw = float(np.mean(q[:,2] - q[:,0]))
    return {"mae": mae, "rmse": rmse, "coverage_q10_q90": cov, "mean_interval_width": iw}

def save_json(obj: Dict, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def plot_loss(history: Dict[str, List[float]], outpath: Path):
    plt.figure()
    plt.plot(history["train_total"], label="train_total")
    plt.plot(history["val_total"], label="val_total")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("Training curve")
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath)
    plt.close()

def plot_pr_curve(y_true: np.ndarray, y_prob: np.ndarray, outpath: Path):
    p, r, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    plt.figure()
    plt.plot(r, p)
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(f"PR curve (AUPRC={ap:.3f})")
    plt.tight_layout()
    plt.savefig(outpath)
    plt.close()

def plot_roc_curve(y_true: np.ndarray, y_prob: np.ndarray, outpath: Path):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    plt.figure()
    plt.plot(fpr, tpr)
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title(f"ROC curve (AUROC={auc:.3f})")
    plt.tight_layout()
    plt.savefig(outpath)
    plt.close()


## 7) Centralized training + full validation evaluation


In [9]:

@dataclass
class ExpConfig:
    W: int = 30
    stride: int = 1
    tau: int = 30
    rul_cap: Optional[int] = None
    epochs: int = 30
    batch_size: int = 128
    lr: float = 1e-3
    weight_decay: float = 1e-4
    hidden: int = 64
    dropout: float = 0.1
    seed: int = 42

def build_pair_map(arrays: Dict[str, np.ndarray]) -> Dict[int, Dict[int, int]]:
    pm = {}
    for idx, (u, c) in enumerate(zip(arrays["unit"], arrays["end_cycle"])):
        pm.setdefault(int(u), {})[int(c)] = int(idx)
    return pm

def sample_pairs(pm: Dict[int, Dict[int,int]], units: np.ndarray, endc: np.ndarray, max_pairs: int):
    pairs = []
    for u, c in zip(units, endc):
        u = int(u); c = int(c)
        nxt = c + 1
        if u in pm and nxt in pm[u]:
            pairs.append((pm[u][c], pm[u][nxt]))
        if len(pairs) >= max_pairs:
            break
    return pairs

def run_centralized(cfg: ExpConfig):
    seed_all(cfg.seed)

    wcfg = WindowConfig(W=cfg.W, stride=cfg.stride, tau=cfg.tau, rul_cap=cfg.rul_cap)
    all_units = sorted(df_train["unit"].unique().tolist())
    tr_units, va_units = split_units_enginewise(all_units, SplitConfig(val_ratio=0.2, seed=cfg.seed))

    train_arrays = make_windows_for_units(df_train, tr_units, wcfg)
    val_arrays   = make_windows_for_units(df_train, va_units, wcfg)

    mu, sd = fit_standardizer(train_arrays["X"])
    train_arrays["X"] = apply_standardizer(train_arrays["X"], mu, sd)
    val_arrays["X"]   = apply_standardizer(val_arrays["X"], mu, sd)

    train_ds = WindowDataset(train_arrays)
    val_ds   = WindowDataset(val_arrays)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False)

    model = DualHeadModel(n_features=len(FEAT_COLS), hidden=cfg.hidden, dropout=cfg.dropout).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    lcfg = LossConfig(lambda_rul=1.0, lambda_fault=1.0, lambda_mono=0.1, focal_gamma=2.0)

    pm = build_pair_map(train_arrays)
    pair_idx = sample_pairs(pm, train_arrays["unit"], train_arrays["end_cycle"], max_pairs=20000)

    history = {"train_total": [], "val_total": []}
    best_val = float("inf")
    best_path = RES_DIR / "metrics" / f"best_model_W{cfg.W}_tau{cfg.tau}.pt"

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        t0 = time.time()
        train_losses = []

        for x, y_rul, y_fault, unit, endc in train_loader:
            x = x.to(DEVICE)
            y_rul = y_rul.to(DEVICE)
            y_fault = y_fault.to(DEVICE)

            out = model(x)
            p = out["p_fault"]
            q = out["q"]

            alpha = batch_alpha_from_imbalance(y_fault)
            loss_fault = focal_loss(p, y_fault, alpha=alpha, gamma=lcfg.focal_gamma)
            loss_rul = pinball_loss(q, y_rul)

            loss_mono = torch.tensor(0.0, device=DEVICE)
            if len(pair_idx) > 0:
                k = min(2048, len(pair_idx))
                sel = np.random.choice(len(pair_idx), size=k, replace=False)
                i0 = torch.tensor([pair_idx[s][0] for s in sel], device=DEVICE)
                i1 = torch.tensor([pair_idx[s][1] for s in sel], device=DEVICE)
                x0 = torch.from_numpy(train_arrays["X"][i0.cpu().numpy()]).to(DEVICE)
                x1 = torch.from_numpy(train_arrays["X"][i1.cpu().numpy()]).to(DEVICE)
                q0 = model(x0)["q"][:,1]
                q1 = model(x1)["q"][:,1]
                loss_mono = monotonic_loss(q0, q1)

            loss = lcfg.lambda_fault * loss_fault + lcfg.lambda_rul * loss_rul + lcfg.lambda_mono * loss_mono

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()

            train_losses.append(float(loss.item()))

        model.eval()
        val_losses = []
        with torch.no_grad():
            for x, y_rul, y_fault, unit, endc in val_loader:
                x = x.to(DEVICE)
                y_rul = y_rul.to(DEVICE)
                y_fault = y_fault.to(DEVICE)

                out = model(x)
                p = out["p_fault"]
                q = out["q"]
                alpha = batch_alpha_from_imbalance(y_fault)
                loss_fault = focal_loss(p, y_fault, alpha=alpha, gamma=lcfg.focal_gamma)
                loss_rul = pinball_loss(q, y_rul)
                loss = lcfg.lambda_fault * loss_fault + lcfg.lambda_rul * loss_rul
                val_losses.append(float(loss.item()))

        trL = float(np.mean(train_losses)) if train_losses else float("nan")
        vaL = float(np.mean(val_losses)) if val_losses else float("nan")
        history["train_total"].append(trL)
        history["val_total"].append(vaL)

        if vaL < best_val:
            best_val = vaL
            torch.save({
                "model_state": model.state_dict(),
                "cfg": asdict(cfg),
                "mu": mu,
                "sd": sd,
                "feat_cols": FEAT_COLS,
                "alphas": ALPHAS,
            }, best_path)

        print(f"Epoch {epoch:03d}/{cfg.epochs} | train {trL:.4f} | val {vaL:.4f} | best {best_val:.4f} | {time.time()-t0:.1f}s")

    ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    y_true_fault, y_prob_fault = [], []
    y_true_rul, y_q = [], []

    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False)
    with torch.no_grad():
        for x, y_rul, y_fault, unit, endc in val_loader:
            out = model(x.to(DEVICE))
            y_true_fault.append(y_fault.numpy().astype(int))
            y_prob_fault.append(out["p_fault"].detach().cpu().numpy())
            y_true_rul.append(y_rul.numpy())
            y_q.append(out["q"].detach().cpu().numpy())

    y_true_fault = np.concatenate(y_true_fault) if y_true_fault else np.array([])
    y_prob_fault = np.concatenate(y_prob_fault) if y_prob_fault else np.array([])
    y_true_rul   = np.concatenate(y_true_rul) if y_true_rul else np.array([])
    y_q          = np.concatenate(y_q) if y_q else np.array([])

    fault_metrics = compute_fault_metrics(y_true_fault, y_prob_fault)
    rul_metrics   = compute_rul_metrics(y_true_rul, y_q)

    metrics = {
        "val": {
            "fault": fault_metrics,
            "rul": rul_metrics,
            "val_loss_best": float(best_val),
            "n_val_windows": int(len(val_ds)),
            "n_val_units": int(len(va_units)),
            "tau": cfg.tau,
            "W": cfg.W,
        },
        "train_units": tr_units,
        "val_units": va_units,
    }

    plot_loss(history, RES_DIR/"plots"/f"loss_W{cfg.W}_tau{cfg.tau}.pdf")
    if len(y_true_fault) and len(np.unique(y_true_fault)) > 1:
        plot_pr_curve(y_true_fault, y_prob_fault, RES_DIR/"plots"/f"pr_val_W{cfg.W}_tau{cfg.tau}.pdf")
        plot_roc_curve(y_true_fault, y_prob_fault, RES_DIR/"plots"/f"roc_val_W{cfg.W}_tau{cfg.tau}.pdf")

    save_json(metrics, RES_DIR/"metrics"/f"metrics_val_W{cfg.W}_tau{cfg.tau}.json")
    return best_path, metrics

cfg = ExpConfig(W=30, tau=30, epochs=30, batch_size=128, lr=1e-3)
best_path, metrics = run_centralized(cfg)
print(json.dumps(metrics["val"], indent=2))


Epoch 001/30 | train 69.3426 | val 52.3326 | best 52.3326 | 13.8s
Epoch 002/30 | train 40.2011 | val 33.1720 | best 33.1720 | 33.6s
Epoch 003/30 | train 33.7539 | val 31.9521 | best 31.9521 | 32.9s
Epoch 004/30 | train 33.0389 | val 30.9901 | best 30.9901 | 32.9s
Epoch 005/30 | train 32.7818 | val 31.5928 | best 30.9901 | 32.1s
Epoch 006/30 | train 32.2851 | val 30.7273 | best 30.7273 | 32.8s
Epoch 007/30 | train 32.0682 | val 30.4372 | best 30.4372 | 31.6s
Epoch 008/30 | train 31.9027 | val 30.2534 | best 30.2534 | 29.3s
Epoch 009/30 | train 31.7115 | val 30.0512 | best 30.0512 | 30.0s
Epoch 010/30 | train 31.5019 | val 29.8786 | best 29.8786 | 30.0s
Epoch 011/30 | train 31.0663 | val 29.1758 | best 29.1758 | 29.7s
Epoch 012/30 | train 30.8568 | val 28.8891 | best 28.8891 | 28.5s
Epoch 013/30 | train 30.8199 | val 28.5808 | best 28.5808 | 29.7s
Epoch 014/30 | train 30.2339 | val 28.9455 | best 28.5808 | 28.9s
Epoch 015/30 | train 29.6193 | val 27.8614 | best 27.8614 | 29.1s
Epoch 016/

C:\Users\gharamik\AppData\Local\Temp\ipykernel_12668\2964368043.py:138: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(best_path, map_location=DEVICE)


{
  "fault": {
    "auprc": 0.9377012906715976,
    "auroc": 0.9868972548176831,
    "f1@0.5": 0.6973580957363327,
    "recall@fpr1%": 0.7036759189797449
  },
  "rul": {
    "mae": 25.37552261352539,
    "rmse": 33.630584716796875,
    "coverage_q10_q90": 0.6079566449070083,
    "mean_interval_width": 49.80599594116211
  },
  "val_loss_best": 25.423228338360786,
  "n_val_windows": 8119,
  "n_val_units": 43,
  "tau": 30,
  "W": 30
}


## 8) Inference on `test.txt` and `final_test.txt` (no labels)


In [10]:

def make_windows_inference(df_raw: pd.DataFrame, mu: np.ndarray, sd: np.ndarray, cfg: WindowConfig) -> Dict[str, np.ndarray]:
    # Build windows per unit; test/final have no RUL, so create dummy RUL column
    df = df_raw.copy()
    unit_ids = sorted(df["unit"].unique().tolist())
    df["RUL"] = 0.0
    arrays = make_windows_for_units(df, unit_ids, cfg)
    arrays["X"] = apply_standardizer(arrays["X"], mu, sd)
    return arrays

def predict_last_per_unit(model: nn.Module, arrays: Dict[str, np.ndarray], batch_size: int = 256) -> pd.DataFrame:
    ds = WindowDataset(arrays)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)

    p_all, q_all = [], []
    unit_all, endc_all = [], []
    with torch.no_grad():
        for x, y_rul, y_fault, unit, endc in loader:
            out = model(x.to(DEVICE))
            p_all.append(out["p_fault"].detach().cpu().numpy())
            q_all.append(out["q"].detach().cpu().numpy())
            unit_all.append(unit.numpy())
            endc_all.append(endc.numpy())

    p_all = np.concatenate(p_all) if p_all else np.array([])
    q_all = np.concatenate(q_all) if q_all else np.array([])
    unit_all = np.concatenate(unit_all) if unit_all else np.array([])
    endc_all = np.concatenate(endc_all) if endc_all else np.array([])

    rows = []
    for u in np.unique(unit_all):
        mask = (unit_all == u)
        idx = np.argmax(endc_all[mask])
        true_idx = np.where(mask)[0][idx]
        rows.append({
            "unit": int(u),
            "end_cycle": int(endc_all[true_idx]),
            "p_fault": float(p_all[true_idx]),
            "q10": float(q_all[true_idx,0]),
            "q50": float(q_all[true_idx,1]),
            "q90": float(q_all[true_idx,2]),
            "rul_hat": float(q_all[true_idx,1]),
        })
    return pd.DataFrame(rows).sort_values("unit").reset_index(drop=True)

ckpt = torch.load(best_path, map_location=DEVICE)
mu = ckpt["mu"]; sd = ckpt["sd"]
wcfg = WindowConfig(W=cfg.W, stride=cfg.stride, tau=cfg.tau, rul_cap=cfg.rul_cap)

model = DualHeadModel(n_features=len(FEAT_COLS), hidden=cfg.hidden, dropout=cfg.dropout).to(DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()

test_arrays  = make_windows_inference(df_test_raw,  mu, sd, wcfg)
final_arrays = make_windows_inference(df_final_raw, mu, sd, wcfg)

df_pred_test  = predict_last_per_unit(model, test_arrays,  batch_size=256)
df_pred_final = predict_last_per_unit(model, final_arrays, batch_size=256)

df_pred_test.to_csv(RES_DIR/"preds"/"test_preds_last_per_unit.csv", index=False)
df_pred_final.to_csv(RES_DIR/"preds"/"final_test_preds_last_per_unit.csv", index=False)

df_pred_test.head()


C:\Users\gharamik\AppData\Local\Temp\ipykernel_12668\4154224566.py:45: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(best_path, map_location=DEVICE)


,unit,end_cycle,p_fault,q10,q50,q90,rul_hat
0,1,54,0.136892,93.960281,165.359909,165.359909,165.359909
1,2,157,0.225507,70.844597,125.374962,125.374962,125.374962
2,3,116,0.239689,62.481865,113.693512,113.693512,113.693512
3,4,74,0.261484,62.038841,113.030441,113.030441,113.030441
4,5,218,0.637645,31.634697,59.255905,59.255905,59.255905


## 9) Diagnostic plots on test/final (PDF)


In [11]:
import matplotlib.pyplot as plt

# Global style (clean IEEE-like)
plt.rcParams.update({
    "font.size": 10,           # base
    "axes.labelsize": 12,      # axis labels (IMPORTANT)
    "axes.titlesize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 300
})

def plot_rul_hist(df_pred, outpath):
    plt.figure(figsize=(3.2, 2.6))  # fits 2 plots in one column
    plt.hist(df_pred["rul_hat"].values, bins=30)
    
    plt.xlabel("Predicted RUL ($q_{50}$)", fontsize=12)
    plt.ylabel("Count", fontsize=12)
    
    plt.tight_layout()
    plt.savefig(outpath, bbox_inches='tight')
    plt.close()

def plot_uncertainty_width(df_pred, outpath):
    width = (df_pred["q90"] - df_pred["q10"]).values
    
    plt.figure(figsize=(3.2, 2.6))
    plt.hist(width, bins=30)
    
    plt.xlabel("Interval width ($q_{90}-q_{10}$)", fontsize=12)
    plt.ylabel("Count", fontsize=12)
    
    plt.tight_layout()
    plt.savefig(outpath, bbox_inches='tight')
    plt.close()


# Generate plots (NO titles)
plot_rul_hist(df_pred_test, RES_DIR/"plots"/"hist_test_rul_q50.pdf")
plot_uncertainty_width(df_pred_test, RES_DIR/"plots"/"hist_test_unc_width.pdf")

plot_rul_hist(df_pred_final, RES_DIR/"plots"/"hist_final_rul_q50.pdf")
plot_uncertainty_width(df_pred_final, RES_DIR/"plots"/"hist_final_unc_width.pdf")

print("Saved to:", (RES_DIR/"plots").resolve())

Saved to: E:\Research\Aircraft_Maintenance\code_PHM2008\results\phm08\plots


In [12]:

# def plot_rul_hist(df_pred: pd.DataFrame, title: str, outpath: Path):
#     plt.figure()
#     plt.hist(df_pred["rul_hat"].values, bins=30)
#     plt.xlabel("Predicted RUL (q50)"); plt.ylabel("Count")
#     plt.title(title)
#     plt.tight_layout()
#     plt.savefig(outpath)
#     plt.close()

# def plot_uncertainty_width(df_pred: pd.DataFrame, title: str, outpath: Path):
#     width = (df_pred["q90"] - df_pred["q10"]).values
#     plt.figure()
#     plt.hist(width, bins=30)
#     plt.xlabel("Interval width (q90-q10)"); plt.ylabel("Count")
#     plt.title(title)
#     plt.tight_layout()
#     plt.savefig(outpath)
#     plt.close()


# plot_rul_hist(df_pred_test, "PHM08 test.txt — predicted RUL (q50)", RES_DIR/"plots"/"hist_test_rul_q50.pdf")
# plot_uncertainty_width(df_pred_test, "PHM08 test.txt — uncertainty width (q90-q10)", RES_DIR/"plots"/"hist_test_unc_width.pdf")

# plot_rul_hist(df_pred_final, "PHM08 final_test.txt — predicted RUL (q50)", RES_DIR/"plots"/"hist_final_rul_q50.pdf")
# plot_uncertainty_width(df_pred_final, "PHM08 final_test.txt — uncertainty width (q90-q10)", RES_DIR/"plots"/"hist_final_unc_width.pdf")

# print("Saved to:", (RES_DIR/"plots").resolve())


## 10) Outputs written
- `results/phm08/plots/` (PDF)
- `results/phm08/metrics/metrics_val_*.json`
- `results/phm08/preds/*.csv`
